# 🎥 TontineSats Demo Video Renderer (Google Colab)

This notebook renders the **TontineSats product demo video** (~2 min 20 sec) in **1080p @ 30 fps** synced with your Google AI Studio voiceover (`tontinesats_demo_vo.mp3`).

### What this notebook does:
1. **Installs required packages**: Node.js dependencies, Playwright Chromium, OpenCV, and FFmpeg.
2. **Launches TontineSats**: Runs the local Vite frontend and LibSQL backend in the background.
3. **Automated Human Recording**: Drives the browser with a visible human arrow cursor, smooth mouse movement, zooms on key interactions (`Landing` → `EN switch` → `Demo circle` → `Diaspora switch` → `Lightning P2P Payment` → `Receipt & Trust Passport` → `Live Circle Creation`).
4. **Composites & Muxes**: Merges the intro animation (`video/scene1-tontine-intro.mp4`), applies camera zooms, muxes the voiceover, and exports `tontinesats-demo.mp4`.
5. **Downloads**: Directly downloads the finished MP4 to your computer.

> **Package requirement:** use the **complete** `tontinesats_colab_package.zip`. Step 1 verifies that `index.html`, `vite.config.ts`, the intro MP4, and the voiceover MP3 are present, and stops with a clear error if any are missing. Without `index.html`, Vite serves a 404 page and Playwright records a blank video.

---


### Step 1: Upload and Extract Project Package
Upload the **complete** `tontinesats_colab_package.zip` (Files panel on the left, or the button below). Any previous extraction is removed first, so reruns cannot mix files from an older, incomplete package.


In [1]:
import os
import shutil
import zipfile
from google.colab import files

ZIP_PATH = "/content/tontinesats_colab_package.zip"
PROJECT_DIR = "/content/tontinesats"
REQUIRED = [
    "index.html",
    "vite.config.ts",
    "package.json",
    "video/scene1-tontine-intro.mp4",
    "tontinesats_demo_vo.mp3",
]

if not os.path.exists(ZIP_PATH):
    print("Please select and upload tontinesats_colab_package.zip:")
    uploaded = files.upload()
    uploaded_zip = next((name for name in uploaded if name.endswith(".zip")), None)
    if not uploaded_zip:
        raise SystemExit("Upload aborted: no .zip file was selected.")
    if os.path.abspath(uploaded_zip) != ZIP_PATH:
        shutil.move(uploaded_zip, ZIP_PATH)

shutil.rmtree(PROJECT_DIR, ignore_errors=True)
os.makedirs(PROJECT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(PROJECT_DIR)

missing = [name for name in REQUIRED if not os.path.exists(os.path.join(PROJECT_DIR, name))]
if missing:
    raise SystemExit(
        "Incomplete package: missing " + ", ".join(missing) + "\n"
        "Use the complete tontinesats_colab_package.zip."
    )

os.chdir(PROJECT_DIR)
print("Extracted to", PROJECT_DIR)
print("Sanity check passed:", ", ".join(REQUIRED))
!ls -la


Please select and upload tontinesats_colab_package.zip:


Saving tontinesats_colab_package.zip to tontinesats_colab_package.zip
/content/tontinesats
total 24
drwxr-xr-x  4 root root 4096 Nov 20 12:11 .
drwxr-xr-x  1 root root 4096 Nov 20 12:11 ..
-rw-rw-rw-  1 root root 1314 Nov 19 19:29 package.json
drwxr-xr-x 11 root root 4096 Nov 20 12:11 src
-rw-rw-rw-  1 root root  595 Nov 19 19:29 tsconfig.json
drwxr-xr-x  3 root root 4096 Nov 20 12:11 video


### Step 2: Install Dependencies
Installs the app npm packages and Playwright Chromium headless browser:

In [3]:
!cd /content/tontinesats && npm install --no-audit --no-fund
!pip install -q playwright opencv-python numpy
!playwright install chromium --with-deps


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 112 packages, and audited 119 packages in 1m
⠋
⠋24 packages are looking for funding
⠋  run `npm fund` for details
⠋
found 0 vulnerabilities
⠋Collecting playwright
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 MB 40.3 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 24.1 MB/s  0:00

### Step 3: Run the Complete Demo Video Pipeline
Starts the local servers, records the human cursor flow, applies camera zoom keyframes, and produces the final MP4. Stale servers from an earlier run are stopped first so the fixed ports are free.


In [4]:
!pkill -f "vite.*--port 5175" || true
!pkill -f "src/server/index.ts" || true
!sleep 1
!cd /content/tontinesats && python3 video/scripts/run_pipeline.py


  TontineSats Demo Video Automated Pipeline

[1/4] Starting Hono + LibSQL backend (port 3001)...
[2/4] Starting Vite frontend (port 5175)...
Waiting for servers to become ready...
Backend server is ONLINE on port 3001!
Vite frontend is ONLINE on port 5175!

[3/4] Running Playwright browser recording...

=== Recording Part A (Landing -> Circle Demo -> Passport: 93.75s) ===
            const b = document.querySelector('button[aria-label="English"]');
            if (!b) return null;
            const r = b.getBoundingClientRect();
            return { cx: r.left + r.width / 2, cy: r.top + r.height / 2 };
        })()
            const b = [...document.querySelectorAll('main button')].find(el => el.textContent.trim() === 'Demo' || el.textContent.trim() === 'Démo');
            if (!b) return null;
            const r = b.getBoundingClientRect();
            return { cx: r.left + r.width / 2, cy: r.top + r.height / 2 };
        })()
            const b = [...document.querySelectorAll('butt

### Step 4: Get the Finished Video
The render is already saved on the Colab VM. `google.colab.files.download()` streams a ~21 MB file through a Colab comm channel and can stall; this cell copies the video to Google Drive instead. If you prefer not to mount Drive, use the **Files panel** (folder icon on the left): `tontinesats` → `tontinesats-demo.mp4` → right-click → **Download**.


In [ ]:
import os
import shutil

video_path = "/content/tontinesats/tontinesats-demo.mp4"
if not os.path.exists(video_path):
    raise SystemExit("Video not found - run Step 3 first.")

size_mb = os.path.getsize(video_path) / (1024 * 1024)
print(f"Video ready: {video_path} ({size_mb:.1f} MB)")

# files.download() pushes the whole file through a comm channel and can
# hang on ~20 MB files. Copy to Drive instead (native, resumable download).
from google.colab import drive
drive.mount("/content/drive")

dest = "/content/drive/MyDrive/tontinesats-demo.mp4"
shutil.copy2(video_path, dest)
print(f"Copied to {dest}")
print("Download it from Google Drive, or use the Files panel.")


Error: tontinesats-demo.mp4 not found. Please check the logs in Step 3.
